In [ ]:
import pandas as pd
import numpy as np
from scipy.sparse import hstack

from tools import tokenize_df, make_encoders

In [3]:
df = pd.read_csv("../datasets/porn_detection/train.csv")
df_test = pd.read_csv("../datasets/porn_detection/test.csv")
df = df.dropna(subset=['title'])
df_test = df_test.dropna(subset=['title'])

In [4]:
tokenize_df(df)
tokenize_df(df_test)

In [5]:
encoder, encoder_wb, encoder_url = make_encoders()
encoder_s, encoder_wb_s, encoder_url_s = make_encoders()

y_train = df["label"]
X_train = hstack([encoder_s.fit_transform(df["lemmatized"]), 
                       encoder_wb_s.fit_transform(df["lemmatized"]),
                       encoder_url_s.fit_transform(df["url"])]).tocsr()
X_test = hstack([encoder_s.transform(df_test["lemmatized"]), 
                      encoder_wb_s.transform(df_test["lemmatized"]),
                      encoder_url_s.transform(df_test["url"])]).tocsr()

print(f"Размеры ТРЕНИРОВОЧНОЙ выборки {X_train.shape}")
print(f"Размеры ТЕСТОВОЙ выборки {X_test.shape}")

Размеры ТРЕНИРОВОЧНОЙ выборки (135308, 479988)
Размеры ТЕСТОВОЙ выборки (165378, 479988)


In [8]:
from sklearn.svm import LinearSVC

params = {'C': 0.5, 'class_weight': "balanced", 'loss': 'squared_hinge', 'max_iter': 1000, 'penalty': 'l2'}

sgd = LinearSVC(**params)
sgd.fit(X_train, y_train)
y_predict = sgd.predict(X_test)
subm = pd.DataFrame({"ID": df_test["ID"], "TARGET": y_predict})
subm.to_csv("final_submission.csv", index=False )